# CityLearn MARL Setup for IPPO and MAPPO

This notebook is a self-contained version of the project files. It now uses the local downloaded CityLearn Challenge 2023 dataset when present.

The default is `citylearn_challenge_2023_phase_3_1`, a six-building phase-3 trace closer to the paper setup than the three-building phase-1 warm-up.

## 1. Dependencies

If imports fail, install the local dependency set first. On this Windows/Python 3.10 machine, the working CityLearn version is `citylearn==2.1.2` because the paper's `v2.2b` line is not cleanly installable here.

In [1]:
# Run this only if the imports below fail.
# %pip install -r requirements-citylearn-local.txt


## 2. Reward Functions

Three reward modes are included:

- `flat_shared`: paper baseline, equal weights and shared district reward.
- `local_individual`: each building gets its own local reward signal.
- `uae_weighted`: shared reward with stronger comfort/cost emphasis for the UAE setting.

In [2]:
"""Reward functions for the Adv AI CityLearn MARL project.

The base paper uses a flat shared multi-objective reward:
    -(0.1 * discomfort + 0.1 * electricity + 0.1 * ramping + 0.1 * solar_penalty)

This module keeps that reward available and adds the two project variants from the
summary: local individual reward and UAE-weighted shaped reward.
"""

from __future__ import annotations

from typing import Dict, List, Mapping, Optional

from citylearn.reward_function import RewardFunction


RewardWeights = Mapping[str, float]


DEFAULT_FLAT_WEIGHTS: Dict[str, float] = {
    "discomfort": 0.1,
    "electricity": 0.1,
    "ramping": 0.1,
    "solar": 0.1,
}


UAE_WEIGHTED_WEIGHTS: Dict[str, float] = {
    "discomfort": 0.35,
    "electricity": 0.30,
    "ramping": 0.15,
    "solar": 0.20,
}


class ProjectReward(RewardFunction):
    """Multi-objective reward used by the IPPO/MAPPO CityLearn setup.

    Parameters
    ----------
    env_metadata:
        CityLearn metadata injected by ``CityLearnEnv``.
    weights:
        Weights for ``discomfort``, ``electricity``, ``ramping`` and ``solar``.
    shared:
        If true, every building receives the same district-level reward. If false,
        each building receives its own local signal.
    """

    def __init__(
        self,
        env_metadata: Optional[Mapping] = None,
        weights: Optional[RewardWeights] = None,
        shared: bool = True,
        **_: object,
    ):
        super().__init__(env_metadata)
        self.weights = dict(DEFAULT_FLAT_WEIGHTS if weights is None else weights)
        self.shared = shared
        self.previous_district_electricity: Optional[float] = None
        self.previous_local_electricity: Optional[List[float]] = None

    def calculate(self, observations: List[Mapping[str, float]]) -> List[float]:
        components = [self._components(o) for o in observations]

        district_electricity = float(sum(c["electricity"] for c in components))
        district_ramping = self._ramping(
            current=district_electricity,
            previous=self.previous_district_electricity,
        )
        self.previous_district_electricity = district_electricity

        local_electricity = [c["electricity"] for c in components]
        if self.previous_local_electricity is None:
            local_ramping = [0.0 for _ in local_electricity]
        else:
            local_ramping = [
                abs(current - previous)
                for current, previous in zip(local_electricity, self.previous_local_electricity)
            ]
        self.previous_local_electricity = local_electricity

        if self.shared:
            district_components = {
                "discomfort": float(sum(c["discomfort"] for c in components)),
                "electricity": district_electricity,
                "ramping": district_ramping,
                "solar": float(sum(c["solar"] for c in components)),
            }
            reward = -self._weighted_sum(district_components)
            return [reward for _ in observations]

        rewards = []
        for i, c in enumerate(components):
            local_components = {**c, "ramping": local_ramping[i]}
            rewards.append(-self._weighted_sum(local_components))

        return rewards

    def reset_memory(self) -> None:
        self.previous_district_electricity = None
        self.previous_local_electricity = None

    def _components(self, observation: Mapping[str, float]) -> Dict[str, float]:
        net_electricity = float(observation.get("net_electricity_consumption", 0.0))
        imported_electricity = max(net_electricity, 0.0)
        solar_generation = max(float(observation.get("solar_generation", 0.0)), 0.0)

        return {
            "discomfort": self._discomfort(observation),
            "electricity": imported_electricity,
            "ramping": 0.0,
            "solar": max(imported_electricity - solar_generation, 0.0),
        }

    def _discomfort(self, observation: Mapping[str, float]) -> float:
        if "average_unmet_cooling_setpoint_difference" in observation:
            return max(float(observation["average_unmet_cooling_setpoint_difference"]), 0.0)

        if "indoor_dry_bulb_temperature_delta" in observation:
            return abs(float(observation["indoor_dry_bulb_temperature_delta"]))

        indoor = float(observation.get("indoor_dry_bulb_temperature", 0.0))
        set_point = float(observation.get("indoor_dry_bulb_temperature_set_point", indoor))
        return abs(indoor - set_point)

    @staticmethod
    def _ramping(current: float, previous: Optional[float]) -> float:
        return 0.0 if previous is None else abs(current - previous)

    def _weighted_sum(self, components: Mapping[str, float]) -> float:
        return float(sum(self.weights[name] * components[name] for name in self.weights))


class FlatSharedReward(ProjectReward):
    """Paper baseline: equal weights and one shared district reward."""

    def __init__(self, env_metadata: Optional[Mapping] = None, **kwargs: object):
        super().__init__(env_metadata, weights=DEFAULT_FLAT_WEIGHTS, shared=True, **kwargs)


class LocalIndividualReward(ProjectReward):
    """Project variant: each building receives its own local KPI signal."""

    def __init__(self, env_metadata: Optional[Mapping] = None, **kwargs: object):
        super().__init__(env_metadata, weights=DEFAULT_FLAT_WEIGHTS, shared=False, **kwargs)


class UAEWeightedSharedReward(ProjectReward):
    """Project variant: stronger comfort/cost emphasis for UAE transfer tests."""

    def __init__(self, env_metadata: Optional[Mapping] = None, **kwargs: object):
        super().__init__(env_metadata, weights=UAE_WEIGHTED_WEIGHTS, shared=True, **kwargs)


REWARD_FUNCTIONS = {
    "flat_shared": FlatSharedReward,
    "local_individual": LocalIndividualReward,
    "uae_weighted": UAEWeightedSharedReward,
}


/Users/maleehafatima/Desktop/MARL/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 3. CityLearn Multi-Agent Environment

This wrapper keeps CityLearn multi-agent instead of flattening everything into one PPO agent.

- IPPO uses `observations[agent]`, `action_spaces[agent]`, and `rewards[agent]`.
- MAPPO uses the same decentralized actor interface plus `env.state()` for the centralized critic.
- Actions are mapped into CityLearn's native action bounds, with optional safety scaling.
- Local schemas are discovered from `citylearn 2023 dataset`.

In [3]:
"""Multi-agent CityLearn adapter for IPPO and MAPPO training."""

from __future__ import annotations

import shutil
from pathlib import Path
from typing import Dict, Iterable, List, Mapping, Optional, Tuple, Union

import gymnasium as gym
import numpy as np
import pandas as pd
from citylearn.citylearn import CityLearnEnv
from citylearn.data import DataSet


Array = np.ndarray
ActionDict = Mapping[str, Array]
ObservationDict = Dict[str, Array]
PROJECT_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd()
LOCAL_DATASET_ROOT = PROJECT_ROOT / "citylearn 2023 dataset"
DEFAULT_PAPER_DATASET = "citylearn_challenge_2023_phase_3_1"


class CityLearnMARLEnv:
    """Parallel multi-agent wrapper around CityLearn.

    IPPO uses ``observations[agent]``, ``action_spaces[agent]`` and
    ``rewards[agent]`` independently. MAPPO uses the same decentralized actor
    interface and additionally calls :meth:`state` to feed a centralized critic.

    Actions are constrained before entering CityLearn. With
    ``normalized_actions=True``, trainers output values in ``[-1, 1]`` and this
    wrapper maps them into each building's native CityLearn action bounds. The
    optional ``action_safety_factor`` shrinks that range around the device
    neutral point while preserving the environment's physical bounds.
    """

    def __init__(
        self,
        dataset_name: str = DEFAULT_PAPER_DATASET,
        reward_mode: str = "flat_shared",
        schema: Optional[Union[str, Path, Mapping]] = None,
        buildings: Optional[Iterable[Union[str, int]]] = None,
        seed: Optional[int] = None,
        episode_time_steps: Optional[Union[int, List[Tuple[int, int]]]] = None,
        normalize_actions: bool = True,
        action_safety_factor: float = 1.0,
        invalid_action_penalty: float = -100.0,
        terminate_on_invalid_action: bool = False,
    ):
        if reward_mode not in REWARD_FUNCTIONS:
            valid = ", ".join(sorted(REWARD_FUNCTIONS))
            raise ValueError(f"Unknown reward_mode={reward_mode!r}. Valid values: {valid}")

        if not 0.0 < action_safety_factor <= 1.0:
            raise ValueError("action_safety_factor must be in the interval (0, 1].")

        self.dataset_name = dataset_name
        self.reward_mode = reward_mode
        self.normalize_actions = normalize_actions
        self.action_safety_factor = float(action_safety_factor)
        self.invalid_action_penalty = float(invalid_action_penalty)
        self.terminate_on_invalid_action = terminate_on_invalid_action
        self._seed = seed

        citylearn_schema = get_citylearn_schema(dataset_name) if schema is None else schema
        self.base_env = CityLearnEnv(
            citylearn_schema,
            buildings=list(buildings) if buildings is not None else None,
            episode_time_steps=episode_time_steps,
            reward_function=REWARD_FUNCTIONS[reward_mode],
            central_agent=False,
            random_seed=seed,
        )

        self.agents = [building.name for building in self.base_env.buildings]
        self.possible_agents = list(self.agents)
        self.native_action_spaces = {
            agent: _to_gymnasium_box(space)
            for agent, space in zip(self.agents, self.base_env.action_space)
        }
        self.observation_spaces = {
            agent: _to_gymnasium_box(space)
            for agent, space in zip(self.agents, self.base_env.observation_space)
        }
        self.action_spaces = self._build_action_spaces()
        self.state_space = self._build_state_space()
        self._last_observations: Optional[ObservationDict] = None
        self._last_info: Dict = {}

    def reset(self, seed: Optional[int] = None) -> Tuple[ObservationDict, Dict[str, Dict]]:
        if seed is not None:
            np.random.seed(seed)

        result = self.base_env.reset()
        if isinstance(result, tuple) and len(result) == 2:
            obs, info = result
        else:
            obs, info = result, {}

        if hasattr(self.base_env.reward_function, "reset_memory"):
            self.base_env.reward_function.reset_memory()

        observations = self._format_observations(obs)
        self._last_observations = observations
        infos = {agent: dict(info or {}) for agent in self.agents}
        return observations, infos

    def step(
        self,
        actions: ActionDict,
    ) -> Tuple[ObservationDict, Dict[str, float], Dict[str, bool], Dict[str, bool], Dict[str, Dict]]:
        action_list = self._format_actions(actions)
        invalid_action = False
        self._last_info = {}

        try:
            result = self.base_env.step(action_list)
        except AssertionError as exc:
            invalid_action = True
            self._last_info = {"invalid_action_error": str(exc)}
            result = self.base_env.step(self._neutral_action_list())

        obs, rewards, terminated, truncated, info = _normalize_step_result(result)
        observations = self._format_observations(obs)
        reward_dict = self._format_rewards(rewards)

        if invalid_action:
            reward_dict = {agent: self.invalid_action_penalty for agent in self.agents}
            if self.terminate_on_invalid_action:
                terminated = True

        self._last_observations = observations
        info = {**self._last_info, **(info or {})}
        infos = {agent: dict(info) for agent in self.agents}
        terminations = {agent: bool(terminated) for agent in self.agents}
        truncations = {agent: bool(truncated) for agent in self.agents}
        return observations, reward_dict, terminations, truncations, infos

    def state(self) -> Array:
        """Return concatenated local observations for MAPPO's centralized critic."""

        if self._last_observations is None:
            raise RuntimeError("Call reset() before requesting MAPPO state().")

        return np.concatenate([self._last_observations[agent] for agent in self.agents]).astype(
            np.float32
        )

    def sample_actions(self) -> Dict[str, Array]:
        return {agent: self.action_spaces[agent].sample() for agent in self.agents}

    def evaluate(self) -> pd.DataFrame:
        return pd.DataFrame(self.base_env.evaluate())

    def selected_district_kpis(self) -> pd.DataFrame:
        kpis = self.evaluate()
        names = [
            "cost_total",
            "carbon_emissions_total",
            "daily_one_minus_load_factor_average",
            "discomfort_hot_delta_average",
            "discomfort_proportion",
            "ramping_average",
        ]
        if {"level", "cost_function", "value"}.issubset(kpis.columns):
            district = kpis[kpis["level"] == "district"]
            return district[district["cost_function"].isin(names)][
                ["cost_function", "value"]
            ].reset_index(drop=True)

        return kpis

    def close(self) -> None:
        close = getattr(self.base_env, "close", None)
        if callable(close):
            close()

    def _build_action_spaces(self) -> Dict[str, gym.spaces.Box]:
        spaces = {}
        for agent, native in self.native_action_spaces.items():
            if self.normalize_actions:
                spaces[agent] = gym.spaces.Box(
                    low=-1.0,
                    high=1.0,
                    shape=native.shape,
                    dtype=np.float32,
                )
            else:
                spaces[agent] = native

        return spaces

    def _build_state_space(self) -> gym.spaces.Box:
        lows = [self.observation_spaces[agent].low for agent in self.agents]
        highs = [self.observation_spaces[agent].high for agent in self.agents]
        return gym.spaces.Box(
            low=np.concatenate(lows).astype(np.float32),
            high=np.concatenate(highs).astype(np.float32),
            dtype=np.float32,
        )

    def _format_observations(self, observations: Iterable) -> ObservationDict:
        formatted = {}
        for agent, obs in zip(self.agents, observations):
            if isinstance(obs, Mapping):
                obs = list(obs.values())
            formatted[agent] = np.asarray(obs, dtype=np.float32).reshape(-1)

        return formatted

    def _format_rewards(self, rewards: Iterable[float]) -> Dict[str, float]:
        rewards = list(rewards)
        if len(rewards) == 1 and len(self.agents) > 1:
            rewards = rewards * len(self.agents)

        return {agent: float(reward) for agent, reward in zip(self.agents, rewards)}

    def _format_actions(self, actions: ActionDict) -> List[Array]:
        missing = [agent for agent in self.agents if agent not in actions]
        if missing:
            raise KeyError(f"Missing actions for agents: {missing}")

        formatted = []
        for agent in self.agents:
            action = np.asarray(actions[agent], dtype=np.float32).reshape(-1)
            native_space = self.native_action_spaces[agent]

            if self.normalize_actions:
                action = np.clip(action, -1.0, 1.0)
                action = self._normalized_to_native(action, native_space)
            else:
                action = np.clip(action, native_space.low, native_space.high)

            formatted.append(action.astype(np.float32))

        return formatted

    def _normalized_to_native(self, action: Array, native_space: gym.spaces.Box) -> Array:
        neutral = _neutral_action(native_space)
        positive_span = native_space.high - neutral
        negative_span = neutral - native_space.low
        scaled = np.where(action >= 0.0, action * positive_span, action * negative_span)
        native = neutral + self.action_safety_factor * scaled
        return np.clip(native, native_space.low, native_space.high)

    def _neutral_action_list(self) -> List[Array]:
        return [_neutral_action(self.native_action_spaces[agent]) for agent in self.agents]


def make_citylearn_marl_env(**kwargs) -> CityLearnMARLEnv:
    """Factory used by training scripts."""

    return CityLearnMARLEnv(**kwargs)


def list_local_citylearn_datasets(root: Union[str, Path] = LOCAL_DATASET_ROOT) -> List[str]:
    """Return dataset names available in the local downloaded CityLearn folder."""

    root = Path(root)
    if not root.exists():
        return []

    return sorted(path.parent.name for path in root.rglob("schema.json"))


def get_citylearn_schema(dataset_name: str = DEFAULT_PAPER_DATASET) -> Mapping:
    """Load a schema from the local downloaded dataset when available.

    Falls back to CityLearn's packaged datasets if the local folder is missing.
    """

    local_schema_path = LOCAL_DATASET_ROOT / dataset_name / "schema.json"
    if local_schema_path.exists():
        return str(local_schema_path)

    return DataSet().get_schema(dataset_name)


def clone_schema_with_temperature_shift(
    dataset_name: str,
    target_directory: Union[str, Path],
    dry_bulb_delta_c: float = 8.0,
) -> Mapping:
    """Clone a packaged CityLearn dataset and make a simple UAE-like hot variant.

    This is a fallback for early zero-shot plumbing tests. For final experiments,
    replace ``weather.csv`` with Dubai/Abu Dhabi TMYx-derived data as described in
    the project summary.
    """

    schema = DataSet().get_schema(dataset_name)
    source = Path(schema["root_directory"])
    target = Path(target_directory)
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(source, target)

    weather_path = target / "weather.csv"
    weather = pd.read_csv(weather_path)
    temperature_columns = [
        column
        for column in weather.columns
        if column == "outdoor_dry_bulb_temperature"
        or column.startswith("outdoor_dry_bulb_temperature_predicted")
    ]
    weather[temperature_columns] = weather[temperature_columns] + dry_bulb_delta_c
    weather.to_csv(weather_path, index=False)

    shifted_schema = DataSet().get_schema(dataset_name)
    shifted_schema["root_directory"] = str(target)
    return shifted_schema


def _to_gymnasium_box(space) -> gym.spaces.Box:
    return gym.spaces.Box(
        low=np.asarray(space.low, dtype=np.float32),
        high=np.asarray(space.high, dtype=np.float32),
        shape=space.shape,
        dtype=np.float32,
    )


def _neutral_action(space: gym.spaces.Box) -> Array:
    zeros = np.zeros(space.shape, dtype=np.float32)
    return np.clip(zeros, space.low, space.high).astype(np.float32)


def _normalize_step_result(result):
    if len(result) == 5:
        obs, rewards, terminated, truncated, info = result
    elif len(result) == 4:
        obs, rewards, done, info = result
        terminated = bool(done)
        truncated = False
    else:
        raise ValueError(f"Unexpected CityLearn step result with {len(result)} values.")

    return obs, rewards, terminated, truncated, info


## 4. Smoke Test

This verifies that all reward modes can reset and step through the local six-building paper dataset.

In [4]:
def run_smoke_test() -> None:
    print("local datasets:", list_local_citylearn_datasets())
    print("default paper dataset:", DEFAULT_PAPER_DATASET)

    for reward_mode in ["flat_shared", "local_individual", "uae_weighted"]:
        env = make_citylearn_marl_env(
            dataset_name=DEFAULT_PAPER_DATASET,
            reward_mode=reward_mode,
            seed=0,
            action_safety_factor=0.75,
        )
        observations, _ = env.reset(seed=0)

        print(f"\nreward_mode={reward_mode}")
        print("agents:", env.agents)
        print("local observation shapes:", {a: o.shape for a, o in observations.items()})
        print("local action spaces:", env.action_spaces)
        print("mappo state shape:", env.state().shape)

        for _ in range(5):
            observations, rewards, terminations, truncations, infos = env.step(env.sample_actions())
            done = any(terminations.values()) or any(truncations.values())
            if done:
                break

        print("last rewards:", rewards)
        print("last info keys:", sorted(next(iter(infos.values())).keys()))
        env.close()


run_smoke_test()


local datasets: []
default paper dataset: citylearn_challenge_2023_phase_3_1

reward_mode=flat_shared
agents: ['Building_1', 'Building_2', 'Building_3', 'Building_4', 'Building_5', 'Building_6']
local observation shapes: {'Building_1': (30,), 'Building_2': (30,), 'Building_3': (30,), 'Building_4': (30,), 'Building_5': (30,), 'Building_6': (30,)}
local action spaces: {'Building_1': Box(-1.0, 1.0, (3,), float32), 'Building_2': Box(-1.0, 1.0, (3,), float32), 'Building_3': Box(-1.0, 1.0, (3,), float32), 'Building_4': Box(-1.0, 1.0, (3,), float32), 'Building_5': Box(-1.0, 1.0, (3,), float32), 'Building_6': Box(-1.0, 1.0, (3,), float32)}
mappo state shape: (180,)
last rewards: {'Building_1': -0.8017456469784066, 'Building_2': -0.8017456469784066, 'Building_3': -0.8017456469784066, 'Building_4': -0.8017456469784066, 'Building_5': -0.8017456469784066, 'Building_6': -0.8017456469784066}
last info keys: []

reward_mode=local_individual
agents: ['Building_1', 'Building_2', 'Building_3', 'Building

## 5. Minimal Usage Pattern

Use this shape when connecting actual IPPO/MAPPO training code.

In [5]:
# Minimal usage pattern for IPPO/MAPPO training code.
# By default this uses the local six-building paper-style dataset:
# citylearn 2023 dataset/citylearn_challenge_2023_phase_3_1/schema.json
env = make_citylearn_marl_env(
    dataset_name=DEFAULT_PAPER_DATASET,
    reward_mode="flat_shared",
    seed=1,
    action_safety_factor=0.75,
)

observations, infos = env.reset(seed=1)

# IPPO: each building/agent receives only its own local observation.
for agent, obs in observations.items():
    print(agent, "local obs shape:", obs.shape, "action shape:", env.action_spaces[agent].shape)

# MAPPO: actors can still be decentralized, but the critic receives this global state.
global_state = env.state()
print("MAPPO centralized critic state shape:", global_state.shape)

# Example one-step rollout with safe sampled actions.
actions = env.sample_actions()
next_obs, rewards, terminations, truncations, infos = env.step(actions)
print("rewards:", rewards)

env.close()


Building_1 local obs shape: (30,) action shape: (3,)
Building_2 local obs shape: (30,) action shape: (3,)
Building_3 local obs shape: (30,) action shape: (3,)
Building_4 local obs shape: (30,) action shape: (3,)
Building_5 local obs shape: (30,) action shape: (3,)
Building_6 local obs shape: (30,) action shape: (3,)
MAPPO centralized critic state shape: (180,)
rewards: {'Building_1': -2.415781494375551, 'Building_2': -2.415781494375551, 'Building_3': -2.415781494375551, 'Building_4': -2.415781494375551, 'Building_5': -2.415781494375551, 'Building_6': -2.415781494375551}


## 6. Optional UAE Temperature-Shift Test

This helper is only for early pipeline testing. The final project should replace weather data with Dubai/Abu Dhabi TMYx-derived profiles for proper zero-shot UAE evaluation.

In [6]:
# Optional: create a simple hotter UAE-like schema for early zero-shot pipeline tests.
# This is NOT the final Dubai/Abu Dhabi TMYx evaluation, but it lets the transfer code path run.
#
# shifted_schema = clone_schema_with_temperature_shift(
#     dataset_name=DEFAULT_PAPER_DATASET,
#     target_directory="citylearn_uae_temperature_shift",
#     dry_bulb_delta_c=8.0,
# )
#
# uae_env = make_citylearn_marl_env(
#     schema=shifted_schema,
#     reward_mode="uae_weighted",
#     seed=1,
#     action_safety_factor=0.75,
# )
# obs, _ = uae_env.reset(seed=1)
# print("UAE-like env agents:", uae_env.agents)
# print("MAPPO state shape:", uae_env.state().shape)
# uae_env.close()
